# Gaining a Statistical Edge in Soccer Prediction using Machine Learning
### Role of Meta-Statistics in Match Prediction

**Team 20** — Kenche Srujana, Krishna Keerthan Reddy D R
**Course:** UE24CS352A — Machine Learning Mini-Project

This notebook compares two feature sets for predicting English Premier League match outcomes
(Home win / Draw / Away win):

1. **Baseline** — simple rolling averages of each team's own raw match stats (goals, shots).
2. **Meta-statistics** — engineered, higher-order features: Elo ratings, weighted recent form,
   head-to-head history, rest days, and betting-market implied probabilities.

The goal is to test whether meta-statistics give a measurable predictive edge over raw stats alone.

> Run top to bottom. No local files needed — data is pulled live from football-data.co.uk.

In [ ]:
!pip install -q scikit-learn pandas numpy matplotlib seaborn requests xgboost

In [ ]:
import requests
import pandas as pd
import numpy as np
from io import StringIO
from collections import defaultdict
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, log_loss, confusion_matrix
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)

## 1. Data Collection

We use free, public match-by-match CSVs from [football-data.co.uk](https://www.football-data.co.uk/)
for the English Premier League (`E0`), seasons 2019/20 through 2024/25 (6 seasons, ~2280 matches).
Each row is one match with full-time result, shot counts, and closing betting odds.

In [ ]:
SEASONS = ["1920", "2021", "2122", "2223", "2324", "2425"]
LEAGUE = "E0"

def download_season(season):
    url = f"https://www.football-data.co.uk/mmz4281/{season}/{LEAGUE}.csv"
    r = requests.get(url, timeout=20)
    r.raise_for_status()
    df = pd.read_csv(StringIO(r.text))
    df["Season"] = season
    return df

frames = [download_season(s) for s in SEASONS]
raw = pd.concat(frames, ignore_index=True, sort=False)
print("Downloaded:", raw.shape)
raw.head()

## 2. Data Cleaning

Keep only the columns we need: match metadata, full-time result, shot counts (for the baseline),
and average closing odds (`AvgH`/`AvgD`/`AvgA`, used to build a market-implied-probability
meta-feature). Parse dates and drop any unplayable rows.

In [ ]:
keep_cols = ["Season", "Date", "HomeTeam", "AwayTeam", "FTHG", "FTAG", "FTR",
             "HS", "AS", "AvgH", "AvgD", "AvgA"]
df = raw[keep_cols].copy()
df["Date"] = pd.to_datetime(df["Date"], dayfirst=True, errors="coerce")
df = df.dropna(subset=["Date", "FTR", "HomeTeam", "AwayTeam"]).sort_values("Date").reset_index(drop=True)

# De-vig the betting odds into a proper implied-probability distribution
inv = 1 / df[["AvgH", "AvgD", "AvgA"]]
overround = inv.sum(axis=1)
df["MktProbH"] = inv["AvgH"] / overround
df["MktProbD"] = inv["AvgD"] / overround
df["MktProbA"] = inv["AvgA"] / overround

print(df.shape)
df.head()

## 3. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df["FTR"].value_counts().reindex(["H", "D", "A"]).plot(
    kind="bar", ax=axes[0], color=["#4C72B0", "#DD8452", "#55A868"])
axes[0].set_title("Match Result Distribution")
axes[0].set_xlabel("Result (H=Home win, D=Draw, A=Away win)")
axes[0].set_ylabel("Count")

df[["FTHG", "FTAG"]].plot(kind="hist", bins=range(0, 8), alpha=0.6, ax=axes[1])
axes[1].set_title("Goals Scored Distribution")
axes[1].set_xlabel("Goals")

plt.tight_layout()
plt.savefig("eda_overview.png", dpi=120)
plt.show()

print("Home win rate:", (df['FTR'] == 'H').mean().round(3))
print("Missing odds rows:", df[["AvgH","AvgD","AvgA"]].isna().any(axis=1).sum())

## 4. Feature Engineering — Baseline vs. Meta-Statistics

**Critical design constraint: no data leakage.** For every match, each feature must only use
information available *before kickoff*. We process matches in chronological order and update each
team's running state (Elo, form, goal difference, head-to-head record) **after** recording that
match's features, never before.

- **Baseline features** — plain rolling averages (last 5 games) of each team's own raw stats:
  goals scored, goal difference, shots.
- **Meta-statistics** — engineered features that summarize broader context:
  - **Elo rating difference** (dynamic team-strength rating updated after every match)
  - **Form difference** (points-per-game over last 5 games)
  - **Goal-difference trend**
  - **Rest-days difference** (fixture congestion)
  - **Head-to-head home win rate**
  - **Market-implied win/draw/loss probability** (the "wisdom of the betting crowd")

In [ ]:
elo = defaultdict(lambda: 1500.0)
last_played = {}
form_pts = defaultdict(list)
gd_hist = defaultdict(list)
gf_hist = defaultdict(list)
shots_hist = defaultdict(list)
h2h = defaultdict(list)

K, HOME_ADV, N = 20, 100, 5

def avg_last_n(buf, n=N):
    return np.mean(buf[-n:]) if buf else np.nan

rows = []
for _, m in df.iterrows():
    home, away, date = m["HomeTeam"], m["AwayTeam"], m["Date"]
    elo_h, elo_a = elo[home], elo[away]

    rest_h = (date - last_played[home]).days if home in last_played else np.nan
    rest_a = (date - last_played[away]).days if away in last_played else np.nan

    key = tuple(sorted([home, away]))
    hist = h2h[key]
    h2h_home_winrate = (sum(1 for w in hist if w == home) / len(hist)) if hist else np.nan

    rows.append({
        "Date": date, "Season": m["Season"], "HomeTeam": home, "AwayTeam": away, "FTR": m["FTR"],
        "AvgGoalsFor_H": avg_last_n(gf_hist[home]), "AvgGoalsFor_A": avg_last_n(gf_hist[away]),
        "AvgGD_H": avg_last_n(gd_hist[home]), "AvgGD_A": avg_last_n(gd_hist[away]),
        "AvgShots_H": avg_last_n(shots_hist[home]), "AvgShots_A": avg_last_n(shots_hist[away]),
        "EloDiff": elo_h - elo_a,
        "FormDiff": avg_last_n(form_pts[home]) - avg_last_n(form_pts[away]),
        "GDDiff": avg_last_n(gd_hist[home]) - avg_last_n(gd_hist[away]),
        "RestDiff": rest_h - rest_a if pd.notna(rest_h) and pd.notna(rest_a) else np.nan,
        "H2HHomeWinRate": h2h_home_winrate,
        "MktProbH": m["MktProbH"], "MktProbD": m["MktProbD"], "MktProbA": m["MktProbA"],
    })

    if m["FTR"] == "H":
        form_pts[home].append(3); form_pts[away].append(0); h2h[key].append(home)
    elif m["FTR"] == "A":
        form_pts[home].append(0); form_pts[away].append(3); h2h[key].append(away)
    else:
        form_pts[home].append(1); form_pts[away].append(1); h2h[key].append("D")

    gd_hist[home].append(m["FTHG"] - m["FTAG"]); gd_hist[away].append(m["FTAG"] - m["FTHG"])
    gf_hist[home].append(m["FTHG"]); gf_hist[away].append(m["FTAG"])
    shots_hist[home].append(m["HS"]); shots_hist[away].append(m["AS"])

    exp_h = 1 / (1 + 10 ** (-((elo_h + HOME_ADV) - elo_a) / 400))
    score_h = 1.0 if m["FTR"] == "H" else (0.5 if m["FTR"] == "D" else 0.0)
    elo[home], elo[away] = elo_h + K * (score_h - exp_h), elo_a + K * ((1 - score_h) - (1 - exp_h))
    last_played[home] = last_played[away] = date

feat_df = pd.DataFrame(rows)
print("Feature matrix:", feat_df.shape)
print("\nMissing value rate (early matches lack history):")
print(feat_df.isna().mean().sort_values(ascending=False).head(6))
feat_df.to_csv("features.csv", index=False)
feat_df.tail()

## 5. Train/Test Split (Chronological)

We train on the first 5 seasons and **test on the most recent season (2024/25)** — a held-out
future period, never using future information to predict the past. This is the only valid split
for time-ordered sports data.

In [ ]:
baseline_feats = ["AvgGoalsFor_H", "AvgGoalsFor_A", "AvgGD_H", "AvgGD_A", "AvgShots_H", "AvgShots_A"]
meta_feats = ["EloDiff", "FormDiff", "GDDiff", "RestDiff", "H2HHomeWinRate", "MktProbH", "MktProbD", "MktProbA"]

train = feat_df[feat_df["Season"] != "2425"].copy()
test = feat_df[feat_df["Season"] == "2425"].copy()
y_train, y_test = train["FTR"], test["FTR"]

print("Train:", train.shape, " Test:", test.shape)
print(train["FTR"].value_counts(normalize=True).round(3))

## 6. Modeling — Baseline vs. Meta-Statistics vs. Combined

Same two classifiers (Logistic Regression, Random Forest) trained three times, on three feature
sets, so any accuracy difference is attributable to the *features*, not the model.

In [ ]:
def run_model(feat_cols, label, results_store):
    imp = SimpleImputer(strategy="median")
    Xtr = imp.fit_transform(train[feat_cols])
    Xte = imp.transform(test[feat_cols])
    scaler = StandardScaler()
    Xtr = scaler.fit_transform(Xtr)
    Xte = scaler.transform(Xte)

    fitted = {}
    for name, clf in [
        ("LogReg", LogisticRegression(max_iter=1000)),
        ("RandForest", RandomForestClassifier(n_estimators=300, max_depth=6, random_state=42)),
    ]:
        clf.fit(Xtr, y_train)
        pred = clf.predict(Xte)
        proba = clf.predict_proba(Xte)
        acc = accuracy_score(y_test, pred)
        f1 = f1_score(y_test, pred, average="macro")
        ll = log_loss(y_test, proba, labels=clf.classes_)
        results_store.append({"FeatureSet": label, "Model": name, "Accuracy": acc, "F1_macro": f1, "LogLoss": ll})
        fitted[name] = clf
    return fitted

results = []
run_model(baseline_feats, "Baseline (raw stats)", results)
meta_models = run_model(meta_feats, "Meta-statistics", results)
run_model(baseline_feats + meta_feats, "Combined", results)

majority = y_train.mode()[0]
maj_acc = (y_test == majority).mean()
results.append({"FeatureSet": "Majority-class", "Model": "-", "Accuracy": maj_acc, "F1_macro": np.nan, "LogLoss": np.nan})

results_df = pd.DataFrame(results)
results_df

In [ ]:
plot_df = results_df[results_df["Model"] != "-"]
fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(data=plot_df, x="FeatureSet", y="Accuracy", hue="Model", ax=ax)
ax.axhline(maj_acc, color="gray", linestyle="--", label=f"Majority-class ({maj_acc:.3f})")
ax.set_title("Test-Set Accuracy by Feature Set (2024/25 season, held out)")
ax.set_ylim(0, 0.65)
ax.legend()
plt.tight_layout()
plt.savefig("model_comparison.png", dpi=120)
plt.show()

## 7. Best Model — Closer Look

Inspect the confusion matrix and feature importances for the Meta-statistics Random Forest,
the strongest-performing feature set.

In [ ]:
best_clf = meta_models["RandForest"]
imp = SimpleImputer(strategy="median").fit(train[meta_feats])
scaler = StandardScaler().fit(imp.transform(train[meta_feats]))
Xte = scaler.transform(imp.transform(test[meta_feats]))
pred = best_clf.predict(Xte)

cm = confusion_matrix(y_test, pred, labels=["H", "D", "A"])
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["H", "D", "A"], yticklabels=["H", "D", "A"], ax=ax)
ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
ax.set_title("Confusion Matrix — Meta-statistics Random Forest")
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=120)
plt.show()

In [ ]:
importances = pd.Series(best_clf.feature_importances_, index=meta_feats).sort_values()
fig, ax = plt.subplots(figsize=(7, 4))
importances.plot(kind="barh", ax=ax, color="#4C72B0")
ax.set_title("Feature Importance — Meta-statistics Random Forest")
ax.set_xlabel("Importance")
plt.tight_layout()
plt.savefig("feature_importance.png", dpi=120)
plt.show()

## 8. Conclusions

- A majority-class guess ("always predict Home win") gets ~41% accuracy — the floor to beat.
- The **raw-stats baseline** (rolling goals/shots averages) only marginally improves on that floor.
- **Meta-statistics** (Elo, form, head-to-head, rest days, market-implied probability) give a
  clear, consistent lift in accuracy, F1, and log-loss over the raw-stats baseline, for both
  Logistic Regression and Random Forest.
- Combining both feature sets does not meaningfully beat meta-statistics alone — the raw rolling
  stats carry little information that Elo/form/market odds don't already capture.
- By feature importance, **market-implied probability** and **Elo rating difference** are the
  strongest single predictors, confirming that aggregated, context-aware statistics ("meta
  statistics") carry a real statistical edge over raw per-match numbers for soccer match
  prediction.

**Next steps (future work):** incorporate player-level data (injuries, lineups), try gradient
boosting (XGBoost/LightGBM) with hyperparameter tuning, and extend to multiple leagues to test
generalization.